# Mapping Exposure - Infraestructure
This notebook explores different sources to obtain georeferenced data about infrastructure

## Open Street Maps - OSM

OpenStreetMap is a free, editable map of the whole world that is being built by volunteers largely from scratch and released with an open-content license.
From this source one can download Points of Interest (POIs) like schools, hospitals, churchs, etc. Also, the entire road network of a city can be downloaded. 
[Source](https://wiki.openstreetmap.org/wiki/About_OpenStreetMap#:~:text=OpenStreetMap%20is%20a%20free%2C%20editable,of%20our%20underlying%20map%20data.)

OSM has an API for accessing to their data. However, this course uses [OSMnx](https://geoffboeing.com/publications/osmnx-paper/), a Python package that simplifies the querying process. 



### Downloading POIs
OSMnx has the feature module that downloads OpenStreetMap geospatial features’ geometries and attributes.
>Retrieve points of interest, building footprints, transit lines/stops, or any other map features from OSM, including their geometries and attribute data, then construct a GeoDataFrame of them. You can use this module to query for nodes, ways, and relations (the latter of type “multipolygon” or “boundary” only) by passing a dictionary of desired OSM tags.

This example will continue working with 02/06/2023 earthquake in Türkiye. The information will be downloaded for a square of 50km around the epicenter.

In [9]:
import osmnx as ox
import geopandas as gpd
from shapely import Point

Considering the epicenter as: 38.011°N 37.196°E:
1. Create a GeoDataFrame for the epicenter
2. Project the point into a projected CRS
3. Generate a buffer of 50km around the epicenter
4. Generate the bounding box to extract the information

In [30]:
d = {'name': ['epicenter'], 'geometry': [Point(37.196, 38.011)]}
gdf = gpd.GeoDataFrame(d, crs="EPSG:4326")
gdf = gdf.to_crs('EPSG:23036')
gdf['buffer'] = gdf.geometry.apply(lambda x: x.buffer(50000))
gdf = gdf.set_geometry('buffer')
gdf = gdf.to_crs('EPSG:4326')

In [25]:
m = gdf.explore()
m

In [31]:
bounds = gdf.bounds
north = bounds.maxy.max()
south = bounds.maxy.min()
west = bounds.minx.min()
east = bounds.maxx.max() 

Acoording to the [documentation](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.features.features_from_bbox), the tags should be specified as follows:
>**tags** (dict) – Dict of tags used for finding elements in the selected area. Results returned are the union, not intersection of each individual tag. Each result matches at least one given tag. The dict keys should be OSM tags, (e.g., building, landuse, highway, etc) and the dict values should be either True to retrieve all items with the given tag, or a string to get a single tag-value combination, or a list of strings to get multiple values for the given tag. For example, tags = {‘building’: True} would return all building footprints in the area. tags = {‘amenity’:True, ‘landuse’:[‘retail’,’commercial’], ‘highway’:’bus_stop’} would return all amenities, landuse=retail, landuse=commercial, and highway=bus_stop.

In order to download all POIS specified as hospitals, one would do the following:

In [43]:
tags = {'amenity': 'hospital'}

In [49]:
pois = ox.features_from_polygon(gdf.geometry.loc[0], tags=tags)
pois #the result is a GeoDataFrame

geometry  \
element_type osmid                                                            
way          323012799    POLYGON ((37.66018 37.79079, 37.66082 37.79013...   
             879054113    POLYGON ((37.24591 38.21331, 37.24659 38.21345...   
node         10738781594                          POINT (36.89587 38.23699)   
way          321950596    POLYGON ((37.15758 38.20408, 37.15762 38.20240...   
             321950616    POLYGON ((37.18443 38.20627, 37.18452 38.20593...   
             641519230    POLYGON ((36.89597 38.23809, 36.89602 38.23778...   

                                                                      nodes  \
element_type osmid                                                            
way          323012799    [2550330093, 2550330094, 2550330095, 255033009...   
             879054113    [8177780901, 8177780902, 3287649682, 817778090...   
node         10738781594                                                NaN   
way          321950596    [3287180177, 3287180178, 6504350304, 650435030...   
             321950616    [3287184311, 3287184312, 3287184313, 328718431...   
             641519230    [6041304479, 6041304480, 6041304481, 604130448...   

                          addr:district   amenity barrier emergency  \
element_type osmid                                                    
way          323012799          Gölbaşı  hospital    wall       yes   
             879054113              NaN  hospital     NaN       NaN   
node         10738781594  Kahramanmaraş  hospital     NaN       NaN   
way          321950596              NaN  hospital     NaN       yes   
             321950616    Kahramanmaraş  hospital     NaN       NaN   
             641519230              NaN  hospital     NaN       NaN   

                         healthcare                           name  \
element_type osmid                                                   
way          323012799     hospital       Gölbaşı Devlet Hastanesi   
             879054113     hospital               Çiçek Sağlık Evi   
node         10738781594   hospital               Devlet Hastanesi   
way          321950596     hospital      Elbistan Devlet Hastanesi   
             321950616     hospital  Ozel Elbistan Yasam Hastanesi   
             641519230     hospital         Afşin Devlet Hastanesi   

                                  operator operator:wikidata  check_date  \
element_type osmid                                                         
way          323012799    Sağlık Bakanlığı          Q4294365         NaN   
             879054113                 NaN               NaN         NaN   
node         10738781594               NaN               NaN  2023-02-07   
way          321950596                 NaN               NaN         NaN   
             321950616                 NaN               NaN  2023-02-07   
             641519230                 NaN               NaN         NaN   

                                            fixme  \
element_type osmid                                  
way          323012799                        NaN   
             879054113                        NaN   
node         10738781594  resurvey after recovery   
way          321950596                        NaN   
             321950616    resurvey after recovery   
             641519230                        NaN   

                                                          note          phone  \
element_type osmid                                                              
way          323012799                                     NaN            NaN   
             879054113                                     NaN            NaN   
node         10738781594           Yeşilyurt Mah. Hastane Cad.  +903445119522   
way          321950596                                     NaN            NaN   
             321950616    Battalgazi Mah. Adnan Menderes Bulv.  +903444130303   
             641519230                                     NaN        

As it is shown, the element type, in this particular case, can be a `way` or a `node` in this case, more information about ways can be found [here](https://wiki.openstreetmap.org/wiki/Way).
The hospitals can be represented as a point or a polygon. 

As it can be seen in the map below, one of the hospital is being double counted. This is expected due to the nature of the data, crowd sourced. Depending on the usage of the data, one might need to clean the dataset with more or less effort. For example, if the goal is to register all the hospitals that might had been affected, then, double-counting might not be an issue. However, if the goal is to estimate number of affected hospitals, double-counting will need to be eliminated. 

In [54]:
pois.explore()

In [57]:
# If the goal is to download all the footprints of buildings, one can avoid specifying a tag and use True
tags = {'building': True}
footprints = ox.features_from_polygon(gdf.geometry.loc[0], tags=tags)

In [59]:
footprints.head()

nodes  \
element_type osmid                                                         
way          96356603  [1116696897, 1116695889, 1116699267, 111669386...   
             96356633  [1116697414, 10657820222, 10657820223, 1116696...   
             96356636  [1116696169, 1116694465, 1116698816, 111669747...   
             96356643  [1116696468, 1116695075, 1116699005, 111669768...   
             96381354  [1116943361, 1116945345, 1116944717, 111694161...   

                      building source  \
element_type osmid                      
way          96356603      yes   bing   
             96356633      yes   bing   
             96356636     roof   bing   
             96356643   mosque   bing   
             96381354      yes   bing   

                                                                geometry  \
element_type osmid                                                         
way          96356603  POLYGON ((37.56554 37.73040, 37.56524 37.73063...   
             96356633  POLYGON ((37.52466 37.79285, 37.52449 37.79282...   
             96356636  POLYGON ((37.50541 37.64448, 37.50507 37.64464...   
             96356643  POLYGON ((37.56499 37.73094, 37.56477 37.73095...   
             96381354  POLYGON ((37.47469 37.79247, 37.47458 37.79242...   

                      layer           amenity religion denomination name  \
element_type osmid                                                         
way          96356603   NaN               NaN      NaN          NaN  NaN   
             96356633   NaN               NaN      NaN          NaN  NaN   
             96356636     1               NaN      NaN          NaN  NaN   
             96356643   NaN  place_of_worship   muslim          NaN  NaN   
             96381354   NaN               NaN      NaN          NaN  NaN   

                      addr:city  ... historic emergency disused:shop  \
element_type osmid               ...                                   
way          96356603       NaN  ...      NaN       NaN          NaN   
             96356633       NaN  ...      NaN       NaN          NaN   
             96356636       NaN  ...      NaN       NaN          NaN   
             96356643       NaN  ...      NaN       NaN          NaN   
             96381354       NaN  ...      NaN       NaN          NaN   

                      townhall:type ruins:building ruins name:nl  fee  \
element_type osmid                                                      
way          96356603           NaN            NaN   NaN     NaN  NaN   
             96356633           NaN            NaN   NaN     NaN  NaN   
             96356636           NaN            NaN   NaN     NaN  NaN   
             96356643           NaN            NaN   NaN     NaN  NaN   
             96381354           NaN            NaN   NaN     NaN  NaN   

                      alt_name name:en  
element_type osmid                      
way          96356603      NaN     NaN  
             96356633      NaN     NaN  
             96356636      NaN     NaN  
             96356643      NaN     NaN  
             96381354      NaN     NaN  

[5 rows x 70 columns]

In [63]:
footprints.amenity.value_counts()

place_of_worship    204
school                9
public_building       5
clinic                4
bank                  4
bus_station           2
car_wash              2
cafe                  2
townhall              2
restaurant            2
police                1
dentist               1
marketplace           1
fire_station          1
courthouse            1
prison                1
mosque                1
Name: amenity, dtype: int64

In [79]:
footprints.explore()

Let's explore the limimations of this data with an example. When downloading all the amenities tagged as `restaurant`, only 8 places are found. It seems unlikely that there are 8 restaurants in the entire region. So, it is worth querying all the amenities. Unfortunately, the result is the same and the quality of the data for restaurants is not good. 

In [75]:
tags = {'amenity': 'restaurant'}
restaurant = ox.features_from_polygon(gdf.geometry.loc[0], tags=tags)

In [76]:
len(restaurant)

8

In [77]:
tags = {'amenity': True}
amenities = ox.features_from_polygon(gdf.geometry.loc[0], tags=tags)

In [78]:
amenities['amenity'].value_counts()

place_of_worship    206
fuel                 84
pharmacy             66
school               45
parking              20
bank                 11
clinic                9
restaurant            8
hospital              6
cafe                  5
public_building       5
atm                   3
townhall              3
car_wash              3
university            2
drinking_water        2
bus_station           2
courthouse            1
mosque                1
dentist               1
police                1
fire_station          1
prison                1
marketplace           1
water_point           1
toilets               1
cinema                1
compressed_air        1
social_facility       1
Name: amenity, dtype: int64

### Debate
- What do you think about this dataset?
- In which crisis related task will it become handy?
- Do you see any other limitation? 
- Which are your proposals to clean this type of data? 

#### Note
OSM data can be downloaded in other ways as well, for example using Qgis or even ChatGPT. This course decided to use OSMnx as it more flexible. 

## Overture 

In [24]:
import pandas as pd
import geopandas as gpd
import overturemaps 
from shapely import wkb
from lonboard import Map, PolygonLayer
from lonboard.colormap import apply_categorical_cmap

import matplotlib.pyplot as plt


In [13]:
# specify bounding box
bbox =  -78.6429, 39.463, -73.7806, 41.6242

In [14]:
# read in Overture Maps land_cover data type
table = overturemaps.record_batch_reader("land_cover", bbox).read_all()
table = table.combine_chunks()

In [15]:
# convert to dataframe
df = table.to_pandas()

In [16]:
df.head()

,id,geometry,bbox,version,update_time,sources,subtype,cartography
0,08b2795602c59fff0005d3229b0b983b,b'\x00\x00\x00\x00\x03\x00\x00\x04\xed\x00\x00...,"{'xmin': -163.012939453125, 'xmax': -55.753906...",0,2024-05-21T22:49:42.122Z,"[{'property': '', 'dataset': 'ESA WorldCover',...",forest,"{'min_zoom': 0, 'max_zoom': 7, 'sort_key': 2}"
1,08b2aa8b36b98fff0005d0035d8a79ba,b'\x00\x00\x00\x00\x03\x00\x00\x00\x02\x00\x00...,"{'xmin': -76.24469757080078, 'xmax': -75.69652...",0,2024-05-21T22:49:42.122Z,"[{'property': '', 'dataset': 'ESA WorldCover',...",crop,"{'min_zoom': 0, 'max_zoom': 7, 'sort_key': 6}"
2,08b2a853132b3fff0005dc1cde8c240d,b'\x00\x00\x00\x00\x03\x00\x00\x00\x03\x00\x00...,"{'xmin': -78.6468734741211, 'xmax': -78.617645...",0,2024-05-21T22:49:42.122Z,"[{'property': '', 'dataset': 'ESA WorldCover',...",grass,"{'min_zoom': 8, 'max_zoom': 15, 'sort_key': 5}"
3,08b2a85310283fff0005d2ec008ebddc,"b""\x00\x00\x00\x00\x03\x00\x00\x00\x01\x00\x00...","{'xmin': -78.62418365478516, 'xmax': -78.61505...",0,2024-05-21T22:49:42.122Z,"[{'property': '', 'dataset': 'ESA WorldCover',...",grass,"{'min_zoom': 8, 'max_zoom': 15, 'sort_key': 5}"
4,08b2a853166d4fff0005db99f065290f,b'\x00\x00\x00\x00\x03\x00\x00\x00\x01\x00\x00...,"{'xmin': -78.66708374023438, 'xmax': -78.625, ...",0,2024-05-21T22:49:42.122Z,"[{'property': '', 'dataset': 'ESA WorldCover',...",grass,"{'min_zoom': 8, 'max_zoom': 15, 'sort_key': 5}"


In [17]:
# filter for higher resolution land_cover features
df_h = df[df.cartography.apply(lambda x: x['min_zoom'] == 8)]

In [18]:
# create color map for land_cover subtypes, loosely based on natural-color palette: https://www.shadedrelief.com/shelton/c.html
color_map = {
    "urban": [167, 162, 186],
    "forest": [134, 178, 137],
    "barren": [245, 237, 213],
    "shrub": [239, 218, 182],
    "grass": [254, 239, 173],
    "crop": [222, 223, 154],
    "wetland": [158, 207, 195], 
    "mangrove": [83, 171, 128],
    "moss": [250, 230, 160],
    "snow": [255, 255, 255],  
}

In [19]:
# apply color map to land_cover subtypes
colors = apply_categorical_cmap(df_h.subtype, color_map)

In [20]:
# dataframe to geodataframe, set crs
gdf = gpd.GeoDataFrame(
    df_h, 
    geometry=df_h['geometry'].apply(wkb.loads), 
    crs="EPSG:4326"
)

In [25]:
gdf.explore()

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x75d568fb9f00>>
Traceback (most recent call last):
  File "/home/sol/venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


TypeError: Object of type ndarray is not JSON serializable

In [28]:
gdf.loc[0:100].explore()

TypeError: Object of type ndarray is not JSON serializable